In [5]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

Envrinoment Varibable Checking

In [7]:
if os.environ.get("GOOGLE_API_KEY"):
    print("api key is found that is gemini")
else:
    raise ValueError("GOOGLE_API_KEY environment variable not set")

api key is found that is gemini


API key validation

In [29]:
from google.genai.errors import APIError
llm_gemini=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
try:
    response = llm_gemini.invoke("Hello")
    print(response.content)
except Exception as e:
    print("Raw Error:", e)

[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'El4KXAERTTIPz3p3c/whQ8v5+0/GVpG2F0jyGL1NJM5f76RK62WDztZ4YTo71+YRemcBMVU+cejXXLs5LrX+4Br/u/CGVvzH0s8osOojcB/3wpXpB9tb9x0fLX4CvndH'}}]


In [34]:
import google.generativeai as genai
import os
for m in genai.list_models():
    if "embedContent" in m.supported_generation_methods:
        print(m.name)

c:\Users\Komalkumar M A\OneDrive\Desktop\LangChain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [37]:
from langchain_community import vectorstores
#step 1 extarcting text from pdfs
from langchain_community.document_loaders.pdf import PyPDFLoader
loader=PyPDFLoader("Docs/langchain_complete_guide.pdf")
docs=loader.load()

#creating own meta data for pdf chunks
for i in docs:
    i.metadata = {
        "source": "langchain_complete_guide.pdf",
        "developer": "claude ai"
    }

# spliting documents ninto chunks
spliter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks=spliter.split_documents(docs)
print(chunks[0])
print(f"\nlength fo chunks :{len(chunks)}\n")
    
# creatign embedding for chunks
embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
result=embeddings.embed_query("what is cricket")
print(result[:5])
print(f"length fo emmdedings :{len(result)}")

#create and store emmbeding in vector store
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

#similarity search

context=vectorstore.similarity_search("explain in detail teh working of retriever")

#talk to llm

response=llm_gemini.invoke("f you can use the following context {context}")



page_content='LangChain: Complete Educational Guide
Building AI-Powered Applications with Language
Models
Author: Technical Documentation
Version: 2.0 (Updated for LangChain 0.1+, LCEL-First)
Date: 2026
Target Audience: Intermediate Python developers building LLM applications
Table of Contents
1. Introduction & Architecture
2. Messages & Message Types
3. LLM Integration
4. Prompts & PromptTemplate
5. Output Parsing
6. Chains & LCEL
7. Structured Output with Pydantic
8. Tools & Tool Binding' metadata={'source': 'langchain_complete_guide.pdf', 'developer': 'claude ai'}

length fo chunks :370

[0.0040961704, 0.014891163, 0.011506862, -0.039000597, -0.03099966]
length fo emmdedings :3072


GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 21.316047631s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', 

locally persist data

In [ ]:
from langchain_community import vectorstores
#step 1 extarcting text from pdfs
from langchain_community.document_loaders.pdf import PyPDFLoader
loader=PyPDFLoader("Docs/langchain_complete_guide.pdf")
docs=loader.load()

#creating own meta data for pdf chunks
for i in docs:
    i.metadata = {
        "source": "langchain_complete_guide.pdf",
        "developer": "claude ai"
    }

# spliting documents ninto chunks
spliter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks=spliter.split_documents(docs)
print(chunks[0])
print(f"\nlength fo chunks :{len(chunks)}\n")
    
# creatign embedding for chunks
embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
result=embeddings.embed_query("what is cricket")
print(result[:5])
print(f"length fo emmdedings :{len(result)}")

#create and store emmbeding in vector store
vectorstore_persist=Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./Vector/"# persiting locally
)
#reuse the vectordb
vectorstore_persist=Chroma(
    persist_directory="./Vector/",# persiting locally
    embedding_function=embeddings
)

#similarity search

context=vectorstore_persist.similarity_search("explain in detail teh working of retriever")

#talk to llm

response=llm_gemini.invoke("f you can use the following context {context}")



page_content='LangChain: Complete Educational Guide
Building AI-Powered Applications with Language
Models
Author: Technical Documentation
Version: 2.0 (Updated for LangChain 0.1+, LCEL-First)
Date: 2026
Target Audience: Intermediate Python developers building LLM applications
Table of Contents
1. Introduction & Architecture
2. Messages & Message Types
3. LLM Integration
4. Prompts & PromptTemplate
5. Output Parsing
6. Chains & LCEL
7. Structured Output with Pydantic
8. Tools & Tool Binding' metadata={'source': 'langchain_complete_guide.pdf', 'developer': 'claude ai'}

length fo chunks :370

[0.0040961704, 0.014891163, 0.011506862, -0.039000597, -0.03099966]
length fo emmdedings :3072


GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 13.474698058s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', 

impelementing rag with multiple docs

In [ ]:
from langchain_community import vectorstores
#step 1 extarcting text from pdfs
from langchain_community.document_loaders.pdf import PyPDFLoader
loader_first=PyPDFLoader("Docs/langchain_complete_guide.pdf")
docs_one=loader_first.load()

# extarcting text from  second pdfs
loader_first=PyPDFLoader("Docs/langchain_quick_reference.pdf")
docs_two=loader_first.load()

#creating own meta data for  first pdf chunks
for i in docs_one:
    i.metadata = {
        "source": "langchain_complete_guide.pdf",
        "developer": "claude ai"
    }

#creating own meta data for second pdf chunks
for i in docs_two:
    i.metadata = {
        "source": "langchain_quick_reference.pdf",
        "developer": "claude ai"
    }

# spliting documents  into chunks
spliter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks_one=spliter.split_documents(docs_one)
print(chunks_one[0])
print(f"\nlength fo chunks :{len(chunks_one)}\n")

chunks_two=spliter.split_documents(docs_two)
print(chunks_two[0])
print(f"\nlength fo chunks :{len(chunks_two)}\n")
    
# creating embedding for chunks
embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
result=embeddings.embed_query("what is cricket")
print(result[:5])
print(f"length fo emmdedings :{len(result)}")

#create and store emmbeding in vector store
vectorstore_persist=Chroma.from_documents(
    documents=chunks_one,
    embedding=embeddings,
    persist_directory="./Vector/"# persiting locally
)
vectorstore_persist.add_documents(chunks_two)
#reuse the vectordb
vectorstore_persist=Chroma(
    persist_directory="./Vector/",# persiting locally
    embedding_function=embeddings
)

#similarity search

context=vectorstore_persist.similarity_search("explain in detail teh working of retriever")

#talk to llm

response=llm_gemini.invoke("f you can use the following context {context}")

